In [ ]:
import json
with open("formula_masks_0.0.json", 'r') as f:
    j _0 = json.load(f)

In [7]:
with open("formula_masks_36.0.json", 'r') as f:
    j_36 = json.load(f)

In [19]:
len(j_0['1']['2']), sum(j_0['3']['2'])

(10000, 963)

In [17]:
import pickle

# Replace with your actual file path
with open('/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run2Full/Masks/0.0%Pruned/ActivationRanges.pkl', 'rb') as file:
    data_0 = pickle.load(file)
with open('/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run2Full/Masks/36.0%Pruned/ActivationRanges.pkl', 'rb') as file:
    data_36 = pickle.load(file)


In [50]:
import torch
clus1 = torch.load("/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run2Full/Masks/0.0%Pruned/Cluster1masks.pt")
clus2 = torch.load("/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run2Full/Masks/0.0%Pruned/Cluster2masks.pt")

clus3 = torch.load("/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run2Full/Masks/0.0%Pruned/Cluster3masks.pt")

In [30]:
avg=0
for i in clus1:
    avg += i.sum()
avg/clus1.shape[0]   

tensor(2425.6191)

In [34]:
clus3.shape[0]  

1024

In [33]:
avg=0
for i in clus3:
    avg += i.sum()
avg/clus3.shape[0]   

tensor(721.9316)

In [ ]:
#on avg clustr1 has more samples activating at each neuron than clu2,3
so fmask has to maximize 1s to maximize iou 
#so at lowest, best form convgs to firmula that civers most if the samples (ie not actuslly explaining convepts learned by the neuron  its covering trying to maximize the range of sentences covered)
# at jhigher activations fewer samples are activ per neuron and fmask overlap with original fmask very low indicating at higher activs the fmask are more specialized
#at lower activ fmask similar throughout meaning its finding formulas covering the same-ish samples depsite pruning but at hgiher its finding formulas that end up covering different samples


# does that mean that pruning leaves the same samples with low/high activations for a given neuron?? 

In [47]:
clus1_36 = torch.load("/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run2Full/Masks/36.0%Pruned/Cluster1masks.pt")
clus2_36 = torch.load("/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run2Full/Masks/36.0%Pruned/Cluster2masks.pt")
clus3_36 = torch.load("/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run2Full/Masks/36.0%Pruned/Cluster3masks.pt")

In [53]:
clus1_36[2],clus2_36[2],clus3_36[2]

(tensor([False,  True, False,  ..., False,  True, False]),
 tensor([False, False, False,  ...,  True, False, False]),
 tensor([False, False, False,  ..., False, False, False]))

In [56]:
clus1[2].sum(),clus2[2].sum(),clus3[2].sum()

(tensor(2385), tensor(1691), tensor(685))

In [55]:
(clus3_36[2] & clus3[2]).sum()/(clus3_36[2] | clus3[2]).sum()

tensor(0.4450)

In [66]:
#hihgest cluster num samples where og and pruned are both activ
ct = 0
cf=0
for i in range(len(clus3_36[2])):
    if clus3[2][i] == clus3_36[2][i] and clus3[2][i] :#neuron2 activ at cluster 3 for og and pruned 
        ct += 1
    elif clus3[2][i] == clus3_36[2][i] and not clus3[2][i]:#neuron2 NOT activ at cluster 3 for og and pruned 
        cf += 1
ct

421

In [65]:
#lowest cluster num samples where og and pruned are both activ
ct = 0
cf=0
for i in range(len(clus1_36[2])):
    if clus1[2][i] == clus1_36[2][i] and clus1[2][i] : #neuron2 activ at cluster 1 for og and pruned 
        ct += 1
    elif clus1[2][i] == clus1_36[2][i] and not clus1[2][i]:#neuron2 NOT activ at cluster 1 for og and pruned 
        cf += 1
ct

1032

In [ ]:
#cluster masks say for each nueron 1 if neuron activated for that sample in that activation range 
# lowest cluster: low overlap means the same neuron is switching activation ranges so if for sent 1 in og 
    #neuron 3 activated at the lowest range, once u prune that neuron 3 now activates for sentence 1 at a hgher range 
#higher overlap in highest cluster meaning more of the strongly activating samples remaingin strongly activating with pruning at that neuron
        #can i graph this (per neuron do the iou like in the cell above for each cluster and avg)
    
            #for neuron in common between 36% and 0%
                #avg += (clus3_36[2] & clus3[2]).sum()/(clus3_36[2] | clus3[2]).sum()
            #avg/(num neurons in common between 36% and 0%)

            

In [ ]:
#at hgher clusters the alignment acorrs pruning is more so that means for each neuron theres more similaritly in the samples that it activates for and despite this the formula masks are vastly different
    #why. well fewer samples activate at high clusters at a given neuron 
    # and pruning doesnt change by much (comp to low clusters) which samples activate at the high value at the neuron
    
    #CORRECTED::::::
        #at the low clusters there a large overlap where og[neuron]==1 and prune[neuron]=1 for each sample. like neuron 2 is active at the lowest range for the same-ish neurons in og as it is in prune
        # at high clusters theres a small overlap (421 vs 1032) meaning formulas that cover more samples need to be different but at low clusters since theres
        # a lot more samples where its active for the neuron despite pruning the formula mask need to cover more of the same samples
        # so if u look at the formula mask at lower activations its going to be more similar and at hgiher its going to be more different 
    
#but at low clusters the alginemnt is so low meaning each neuron activates at the low range for completely different samples yet fmask is so similar
    #measnt hat fmask is favoritn fromuals that just entail as many concepts thats common?